# Training

## Setup stage

Mounting Google Drive.

In [15]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


Cloning git repository to access the codebase.

In [2]:
!git clone -b feat/early-stopping --single-branch https://github.com/deadPixelsGreta/xAI-proj-m-ws2526.git

Cloning into 'xAI-proj-m-ws2526'...
remote: Enumerating objects: 527, done.
remote: Counting objects: 100% (137/137), done.
remote: Compressing objects: 100% (40/40), done.
remote: Total 527 (delta 109), reused 105 (delta 97), pack-reused 390 (from 1)
Receiving objects: 100% (527/527), 132.10 MiB | 36.85 MiB/s, done.
Resolving deltas: 100% (224/224), done.


In [11]:
!git pull

remote: Enumerating objects: 13, done.
remote: Counting objects: 100% (13/13), done.
remote: Compressing objects: 100% (4/4), done.
remote: Total 7 (delta 4), reused 6 (delta 3), pack-reused 0 (from 0)
Unpacking objects: 100% (7/7), 592 bytes | 84.00 KiB/s, done.
From https://github.com/deadPixelsGreta/xAI-proj-m-ws2526
   ca9bb3a..4da1349  feat/early-stopping -> origin/feat/early-stopping
Updating ca9bb3a..4da1349
Fast-forward
 experiments/bagging/src/models/architectures.py | 2 +-
 1 file changed, 1 insertion(+), 1 deletion(-)


Copying dataset from the Google Drive to the current local disk of VM.

In [3]:
import shutil
import os
from tqdm import tqdm

# TODO: changing default path in yaml to "datasets"

source_path = "/content/drive/MyDrive/ImageNetSubset/"
destination_path = "/content/xAI-proj-m-ws2526/datasets/"

# If the destination directory exists, remove it first
if os.path.exists(destination_path):
    print(f"Removing existing directory: {destination_path}")
    shutil.rmtree(destination_path)

# Custom copy function with tqdm
def copytree_with_tqdm(src, dst):
    # Calculate total number of items (files and directories) to copy for tqdm
    total_items = 0
    for dirpath, dirnames, filenames in os.walk(src):
        total_items += len(dirnames) # for directories
        total_items += len(filenames) # for files

    # Ensure the destination root directory exists
    os.makedirs(dst, exist_ok=True)

    with tqdm(total=total_items, unit="item", desc=f"Copying {os.path.basename(src)}") as pbar:
        for dirpath, dirnames, filenames in os.walk(src):
            # Create subdirectories in destination
            relative_path = os.path.relpath(dirpath, src)
            current_dst_dir = os.path.join(dst, relative_path)

            for dirname in dirnames:
                dest_dir = os.path.join(current_dst_dir, dirname)
                os.makedirs(dest_dir, exist_ok=True)
                pbar.update(1)

            # Copy files
            for filename in filenames:
                src_file = os.path.join(dirpath, filename)
                dst_file = os.path.join(current_dst_dir, filename)
                shutil.copy2(src_file, dst_file)
                pbar.update(1)

# Call the custom copy function
copytree_with_tqdm(source_path, destination_path)

Copying : 100%|██████████| 13554/13554 [09:30<00:00, 23.75item/s]


Setup the root directory for the project. IMPORTANT for module imports.

In [6]:
import sys, os
from pathlib import Path

def find_project_root(start: Path) -> Path:
    """Walk upward to find the outermost folder containing common project markers."""
    markers = {".git", "requirements.txt", "setup.py", "pyproject.toml"}
    root = None
    for parent in [start, *start.parents]:
        if any((parent / m).exists() for m in markers):
            root = parent  # keep going to prefer the outermost match
    return root or start

# Dynamically get the name of the cloned repository if it exists
cloned_repo_name = "xAI-proj-m-ws2526"
cloned_repo_path = Path.cwd() / cloned_repo_name

# If the cloned repository exists as a subdirectory, change into it
if cloned_repo_path.is_dir():
    os.chdir(cloned_repo_path)

# Now, find the project root from within the repository (or its parent if already there)
ROOT = find_project_root(Path.cwd()).resolve()

# Ensure we are in the identified project root
os.chdir(ROOT)

if str(ROOT) not in sys.path:
    sys.path.append(str(ROOT))
print("cwd:", Path.cwd())
print("root on sys.path:", str(ROOT) in sys.path)

cwd: /content/xAI-proj-m-ws2526
root on sys.path: True


---

# Only GU-Windows-Pool

In [ ]:
# Initialize conda for PowerShell
# Do this in the TERMINAL
& "C:\ProgramData\anaconda3\Scripts\conda.exe" init powershell

SyntaxError: invalid syntax (ipython-input-2789236835.py, line 3)

In [ ]:
!conda create -n xai-proj python=3.11 -y

---

In [7]:
# Install dependencies
!pip install -r experiments/bagging/requirements.txt --quiet

In [8]:
import wandb

# Login to WandB - this will prompt you to enter your API key
wandb.login()

/usr/local/lib/python3.12/dist-packages/notebook/notebookapp.py:191: SyntaxWarning: invalid escape sequence '\/'
  | |_| | '_ \/ _` / _` |  _/ -_)
wandb: (1) Create a W&B account
wandb: (2) Use an existing W&B account
wandb: (3) Don't visualize my results
wandb: Enter your choice:

 2


wandb: You chose 'Use an existing W&B account'
wandb: Logging into https://api.wandb.ai. (Learn how to deploy a W&B server locally: https://wandb.me/wandb-server)
wandb: Find your API key here: https://wandb.ai/authorize
wandb: Paste an API key from your profile and hit enter:

 ··········


wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: greta-hunshback (shady-university-of-bamberg) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


True

In [ ]:
Use this path on Windows: C:\Users\ba081274\Downloads\ImageNetSubset\ImageNetSubset\

## Training Stage

In [ ]:
!python -m experiments.bagging.scripts.train --config experiments/bagging/configs/default.yaml

RESNET34 Training on ImageNetSubset

Random seed: 0
Device: CUDA (NVIDIA A100-SXM4-40GB)

 Dataset Summary:
   Training samples: 13032
   Validation samples: 500
   Classes: ['binder', 'coffee_mug', 'computer_keyboard', 'mouse', 'notebook', 'remote_control', 'soup_bowl', 'teapot', 'toilet_tissue', 'wooden_spoon']
   Number of classes: 10

Loading pretrained resnet34 weights...
resnet34 ready with 10 output classes

 Training Configuration:
   Epochs: 30
   Batch size: 64
   Learning rate: 0.001
   Momentum: 0.9
   Weight decay: 0.0001
   Pretrained: True
   Save directory: experiments/checkpoints
   Wandb logging: True
wandb: Currently logged in as: greta-hunshback (shady-university-of-bamberg) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin
wandb: ⢿ Waiting for wandb.init()...
wandb: ⣻ Waiting for wandb.init()...
wandb: ⣽ Waiting for wandb.init()...
wandb: Tracking run with wandb version 0.23.1
wandb: Run data is saved locally in wandb/wandb/run-20251222_113921-3

# Sweep

In [12]:
!wandb sweep experiments/bagging/configs/sweep_learning_rate.yaml --project resnet

wandb: Creating sweep from: experiments/bagging/configs/sweep_learning_rate.yaml
wandb: Creating sweep with ID: d9ecbw9d
wandb: View sweep at: https://wandb.ai/shady-university-of-bamberg/resnet/sweeps/d9ecbw9d
wandb: Run sweep agent with: wandb agent shady-university-of-bamberg/resnet/d9ecbw9d


In [13]:
# add the agent id here and --count 50 for roundes
!wandb agent shady-university-of-bamberg/resnet/d9ecbw9d


wandb: Starting wandb agent 🕵️
2025-12-28 22:17:53,750 - wandb.wandb_agent - INFO - Running runs: []
2025-12-28 22:17:53,988 - wandb.wandb_agent - INFO - Agent received command: run
2025-12-28 22:17:53,989 - wandb.wandb_agent - INFO - Agent starting run with config:
	batch_size: 64
	config: experiments/bagging/configs/default.yaml
	lr: 0.053572732920351446
	model: resnet34
	seed: 0
2025-12-28 22:17:53,990 - wandb.wandb_agent - INFO - About to run command: /usr/bin/env python -m experiments.bagging.scripts.train --batch_size=64 --config=experiments/bagging/configs/default.yaml --lr=0.053572732920351446 --model=resnet34 --seed=0
2025-12-28 22:17:58,996 - wandb.wandb_agent - INFO - Running runs: ['vt3pgdxx']
RESNET34 Training on ImageNetSubset

Random seed: 0
Device: CPU

 Dataset Summary:
   Training samples: 13032
   Validation samples: 500
   Classes: ['binder', 'coffee_mug', 'computer_keyboard', 'mouse', 'notebook', 'remote_control', 'soup_bowl', 'teapot', 'toilet_tissue', 'wooden_spo

## Validation Stage

In [ ]:
# Run ensemble evaluation on a dataset
!python -m experiments.scripts.inference --evaluate --data-dir datasets --wandb

In [ ]:
# Upload a test image or use sample
!python -m experiments.scripts.inference --image datasets/test_image.jpg --show-individual

---

## Script for saving best models in Google Drive

In [14]:
# Define source and destination paths
# We check experiments/bagging/checkpoints (relative to project root) first
source_dir = Path("experiments/bagging/checkpoints")

# Check for fallback paths if the default doesn't exist (e.g., if strictly using /chechpoints)
if not source_dir.exists():
    if Path("/chechpoints").exists(): # Handling the specific path mentioned
        source_dir = Path("/chechpoints")
    elif Path("/checkpoints").exists(): # Handling potential typo correction
        source_dir = Path("/checkpoints")

# Destination folder on the mounted drive
# You can change "saved_checkpoints" to your preferred folder name
dest_dir = Path("/content/drive/MyDrive/saved_checkpoints")

print(f"Source Directory: {source_dir}")
print(f"Destination Directory: {dest_dir}")

# Create destination directory if it doesn't exist
os.makedirs(dest_dir, exist_ok=True)

# Copy .pth files
if source_dir.exists():
    pth_files = list(source_dir.glob("*.pth"))

    if not pth_files:
        print("No .pth files found in source directory.")
    else:
        print(f"Found {len(pth_files)} .pth files to copy.")

        for file_path in pth_files:
            try:
                shutil.copy2(file_path, dest_dir / file_path.name)
                print(f"Successfully copied: {file_path.name}")
            except Exception as e:
                print(f"Error copying {file_path.name}: {e}")
else:
    print(f"Source directory {source_dir} not found. Please check the path.")

Source Directory: experiments/bagging/checkpoints
Destination Directory: /content/drive/MyDrive/saved_checkpoints
Source directory experiments/bagging/checkpoints not found. Please check the path.
